# 03 — RAG (Retrieval-Augmented Generation) Sistemi

**İlan maddesi:** *"Vektör veri tabanları, embedding modelleri ve belge yönetimi
bileşenlerini bir araya getirerek RAG mimarileri tasarlamak, geliştirmek ve optimize
etmek."*

Bu notebook'ta:
1. Chunk'ları embedding'e çevirip Chroma'ya indeksliyoruz
2. Retrieval + reranking'i deniyoruz
3. Uçtan uca RAG pipeline'ını (retrieve → prompt → generate) çalıştırıyoruz
4. Diyalog yöneticisini (02. notebook) artık gerçek RAG ile test ediyoruz

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. İndeksleme

In [ ]:
import json
from src.rag.vector_store import index_chunks

chunks = []
with open("data/processed/chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

n = index_chunks(chunks)
print(f"{n} chunk vektör veritabanına indekslendi.")


## 2. Retrieval + Reranking

In [ ]:
from src.rag.retriever import retrieve

results = retrieve("Bayraktar TB2'nin özellikleri nelerdir?", final_k=5)
for r in results:
    print(f"[{r.get('rerank_score', r['similarity']):.3f}] {r['doc_title']} — {r['text'][:100]}...")


## 3. Uçtan uca RAG

In [ ]:
from src.rag.rag_pipeline import answer

result = answer("Baykar hangi tür insansız hava araçları üretir?")
print("CEVAP:\n", result["answer"])
print("\nKAYNAKLAR:")
for s in result["sources"]:
    print("-", s["title"], s["url"])


## 4. Diyalog üzerinden RAG

In [ ]:
from src.nlp_tasks.dialogue import chat

r1 = chat("demo-session", "Bayraktar TB2 nedir?")
print(r1["answer"])

r2 = chat("demo-session", "Peki onu kim geliştirdi?")
print(r2["answer"])
